In [5]:
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window


spark = (
    SparkSession.builder
        .appName("etl-datalake")
        .master("local[*]")
        .getOrCreate()
)

spark.sparkContext.setLogLevel("WARN")

In [6]:
df_hist = spark.read.parquet("data_lake/dim_pessoa_historica")

In [7]:
df_dim_pessoa_hist = df_hist.select(
    F.col("cpf"),
    F.col("nome"),
    F.col("endereco"),
    F.col("data_admissao"),
    F.col("salario"),
    F.col("origem_registro"),
    F.col("data_inicio"),
    F.col("data_fim"),
    F.col("is_current"),
    F.col("data_ingestao")
)

In [8]:
df_dim_pessoa_atual = df_dim_pessoa_hist.filter(
    F.col("is_current") == 1
)

In [9]:
df_dim_pessoa_atual.show(5, False)

+-----------+--------------+----------+-------------+-------+---------------+-----------+--------+----------+-------------+
|cpf        |nome          |endereco  |data_admissao|salario|origem_registro|data_inicio|data_fim|is_current|data_ingestao|
+-----------+--------------+----------+-------------+-------+---------------+-----------+--------+----------+-------------+
|87878787802|Amanda Letícia|Rua H, 800|2019-12-04   |5300   |esocial        |2023-03-01 |NULL    |1         |2026-04-02   |
|66666666606|Ana Lima      |Rua F, 600|2021-07-01   |4000   |esocial        |2023-03-01 |NULL    |1         |2026-04-02   |
|11111111101|João Silva    |Rua A, 120|2020-01-10   |5100   |gestao         |2023-05-01 |NULL    |1         |2026-04-02   |
|77777777707|Bruno Costa   |Rua G, 700|2020-09-10   |3900   |esocial        |2023-03-01 |NULL    |1         |2026-04-02   |
|44444444404|Fernanda Lima |Rua D, 400|2022-02-05   |4700   |gestao         |2023-05-01 |NULL    |1         |2026-04-02   |
+-------

In [10]:
df_dim_pessoa_atual = df_dim_pessoa_atual.withColumn(
    "person_sk",
    F.monotonically_increasing_id()
)

In [13]:
df_dim_pessoa_atual.show(5, False)

+-----------+--------------+----------+-------------+-------+---------------+-----------+--------+----------+-------------+----------+
|cpf        |nome          |endereco  |data_admissao|salario|origem_registro|data_inicio|data_fim|is_current|data_ingestao|person_sk |
+-----------+--------------+----------+-------------+-------+---------------+-----------+--------+----------+-------------+----------+
|87878787802|Amanda Letícia|Rua H, 800|2019-12-04   |5300   |esocial        |2023-03-01 |NULL    |1         |2026-04-02   |0         |
|66666666606|Ana Lima      |Rua F, 600|2021-07-01   |4000   |esocial        |2023-03-01 |NULL    |1         |2026-04-02   |1         |
|11111111101|João Silva    |Rua A, 120|2020-01-10   |5100   |gestao         |2023-05-01 |NULL    |1         |2026-04-02   |2         |
|77777777707|Bruno Costa   |Rua G, 700|2020-09-10   |3900   |esocial        |2023-03-01 |NULL    |1         |2026-04-02   |8589934592|
|44444444404|Fernanda Lima |Rua D, 400|2022-02-05   |47

In [12]:
########## Salva no Baco de Dados (PostgreSQL) - IGNORE ########## - Relacional - Pode ser MySQL, SQL Server, etc.
# url = "jdbc:postgresql://localhost:5432/volpy_db"

# properties = {
#     "user": "seu_usuario",
#     "password": "sua_senha",
#     "driver": "org.postgresql.Driver"

# }

# df_dim_pessoa_atual.write \
#     .jdbc(
#         url=url,
#         table="dim_pessoa_atual",
#         mode="overwrite",
#         properties=properties
#     )